In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [33]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
start_date = '20241001'
end_date = '20241031'
retention_start_date = '20241101'
retention_end_date = '20241231'

In [4]:
query_data = f'''

    WITH customer_base AS (
    
    SELECT 
        distinct
        -- yyyymmdd,
        city_name,
        -- service_obj_service_name,
        customer_id
        -- order_id
    FROM 
        orders.order_logs_fact
    WHERE 
        yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND city_name IN ('Bangalore', 'Chennai', 'Hyderabad')
        AND order_status = 'dropped'
        AND service_obj_service_name in ('Auto','Auto Pool','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    ),
    
    
    unlocked AS (

    SELECT 
        distinct
        -- kpsi.yyyymmdd,
        kpsi.userid as user_id,
        kpsi.rewards__transactionid as rewards_transaction_id
    FROM 
        hive.canonical.iceberg_domain_pricing_scratchcards_immutable kpsi
    WHERE 
        yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND rewards__type = 'EXTERNAL_REWARD'
        AND rewards__state = 'UNLOCKED'
    ),
    
    scratched AS (

    SELECT 
        distinct
        -- kpsi.yyyymmdd,
        date_format(date_parse(kpsi.yyyymmdd, '%Y%m%d')  + interval '7' day, '%Y%m%d') scratch_window,
        kpsi.userid as user_id,
        kpsi.rewards__transactionid as rewards_transaction_id
    FROM 
        hive.canonical.iceberg_domain_pricing_scratchcards_immutable kpsi
    WHERE 
        yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND rewards__type = 'EXTERNAL_REWARD'
        AND rewards__state = 'SCRATCHED'
    ),
    
    redeemed AS (
    
    SELECT 
        distinct
        event_props_user_id redeem_user_id
    FROM 
        clevertap.customer_rewards_immutable 
    WHERE 
       yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND event_props_action NOT IN ('ctaClicked', 'copyCodeClicked')
    
    ),
    
    merge AS (
    
    SELECT 
        customer_base.*,
        unlocked.rewards_transaction_id unlocked,
        scratched.rewards_transaction_id scratched,
        redeem_user_id
    FROM 
        customer_base
    LEFT JOIN 
        unlocked
        -- ON customer_base.yyyymmdd = unlocked.yyyymmdd
        ON customer_base.customer_id = unlocked.user_id
        -- AND customer_base.order_id = unlocked.rewards_transaction_id
    LEFT JOIN 
        scratched
        -- ON unlocked.yyyymmdd <= scratched.scratch_window
        ON unlocked.user_id = scratched.user_id
        -- AND unlocked.rewards_transaction_id = scratched.rewards_transaction_id
    LEFT JOIN 
        redeemed
        ON scratched.user_id = redeemed.redeem_user_id
    ),
    
    view AS (
    
    SELECT
        city_name,
        CASE 
        WHEN unlocked = 0 THEN 'Deals Not Issued'
        WHEN unlocked > 0 THEN 'Deals issused'
        -- WHEN unlocked > 0 AND scratched = 0 THEN 'Deals issused'
        -- WHEN unlocked > 0 AND scratched > 0 THEN 'GROUP-B'
        ELSE 'Check'
        END isIssued,
        CASE 
        WHEN unlocked = 0 THEN 'Deals Not Issued'
        WHEN unlocked > 0 AND scratched = 0 THEN 'Deals issused, not scratched'
        WHEN unlocked > 0 AND scratched > 0 THEN 'Deals issused, scratched'
        ELSE 'Check'
        END isScratched,
        CASE 
        WHEN unlocked = 0 THEN 'Deals Not Issued'
        WHEN unlocked > 0 AND scratched = 0 THEN 'Deals issused, not scratched'
        WHEN unlocked > 0 AND scratched > 0 AND redeemed = 0  THEN 'Deals issused, scratched not redeemed'
        WHEN unlocked > 0 AND scratched > 0 AND redeemed > 0  THEN 'Deals issused, redeemed'
        ELSE 'Check'
        END isredeemed,
        customer_id,
        unlocked,
        scratched,
        redeemed
    FROM
        (
        SELECT 
            city_name,
            customer_id,
            count(distinct unlocked) unlocked,
            count(distinct scratched) scratched,
            COUNT(distinct redeem_user_id) redeemed
        FROM 
            merge
        GROUP BY 1,2
        )
    ),
    
    customer_retention AS (
    
    SELECT 
        customer_id,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241101' THEN 1 ELSE 0 END) nov_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241201' THEN 1 ELSE 0 END) dec_retention
    FROM 
        orders.order_logs_fact
    WHERE 
        yyyymmdd >= '{retention_start_date}' 
        AND yyyymmdd <= '{retention_end_date}'
        AND city_name IN ('Bangalore', 'Chennai', 'Hyderabad')
        AND order_status = 'dropped'
        AND service_obj_service_name in ('Auto','Auto Pool','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    GROUP BY 1
    ),
    
    segment AS (
    
    SELECT  
        customer_id,
        taxi_ltr_segment,
        taxi_retention_segments,
        taxi_regularity_segment,
        taxi_fe_regularity_segment,
        taxi_recency_segment
    FROM 
        datasets.iallocator_customer_segments
    WHERE
        run_date =  '2024-10-31' 
        AND taxi_recency_segment != 'INACTIVE'
    )
    
    SELECT 
        view.*,
        coalesce(nov_retention, 0) nov_retention,
        coalesce(dec_retention, 0) dec_retention,
        taxi_ltr_segment,
        taxi_retention_segments,
        taxi_regularity_segment,
        taxi_fe_regularity_segment,
        taxi_recency_segment
    FROM 
        view
    LEFT JOIN 
        customer_retention
        ON view.customer_id = customer_retention.customer_id
    LEFT JOIN 
        segment
        ON view.customer_id = segment.customer_id
'''

df = pd.read_sql(query_data, connection)

In [5]:
df.head(5)

,city_name,isIssued,isScratched,isredeemed,customer_id,unlocked,scratched,redeemed,nov_retention,dec_retention,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_fe_regularity_segment,taxi_recency_segment
0,Hyderabad,Deals issused,"Deals issused, not scratched","Deals issused, not scratched",6718ee679f872a0216b733ff,1,0,0,0,0,HH,HH,UNKNOWN,MONTHLY,STATIONARY
1,Hyderabad,Deals issused,"Deals issused, not scratched","Deals issused, not scratched",603fa2dc44b59ebb30d89308,1,0,0,1,1,PHH,PLATINUM,WEEKLY,WEEKLY,STATIONARY
2,Hyderabad,Deals Not Issued,Deals Not Issued,Deals Not Issued,5ded507d1a882e3354b957a8,0,0,0,0,0,PHH,DORMANT,BI_WEEKLY,MONTHLY,DORMANT
3,Hyderabad,Deals Not Issued,Deals Not Issued,Deals Not Issued,62033923725c4c4025b05f33,0,0,0,1,1,PHH,GOLD,WEEKLY,BI_WEEKLY,STATIONARY
4,Hyderabad,Deals issused,"Deals issused, not scratched","Deals issused, not scratched",5c2b81f04a267149c762c9a0,1,0,0,0,0,PHH,GOLD,BI_WEEKLY,BI_WEEKLY,STATIONARY


In [6]:
df.to_parquet('/Users/E2074/local-datasets/customer/ads-monetisation/deals-impact/data.parquet', index=False)

### Analysis

In [7]:
df_analysis1 = df\
                .groupby(['city_name'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis1['nov retention %'] = round(df_analysis1['nov_retention']*100.00/df_analysis1['customers'])
df_analysis1['dec retention %'] = round(df_analysis1['dec_retention']*100.00/df_analysis1['customers'])
df_analysis1

,city_name,customers,nov_retention,dec_retention,nov retention %,dec retention %
0,Bangalore,2637619,1665721,1611666,63.0,61.0
1,Chennai,1787845,1130271,1079362,63.0,60.0
2,Hyderabad,3478486,2211408,2159751,64.0,62.0


In [26]:
df_analysis2 = df\
                .groupby(['city_name','isIssued'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis2['customers distr %'] = round(df_analysis2['customers']*100.00/df_analysis2.groupby(['city_name'])['customers'].transform('sum'), 2)
df_analysis2['nov retention %'] = round(df_analysis2['nov_retention']*100.00/df_analysis2['customers'])
df_analysis2['dec retention %'] = round(df_analysis2['dec_retention']*100.00/df_analysis2['customers'])
df_analysis2

,city_name,isIssued,customers,nov_retention,dec_retention,customers distr %,nov retention %,dec retention %
0,Bangalore,Deals Not Issued,522616,254753,251422,19.81,49.0,48.0
1,Bangalore,Deals issused,2115003,1410968,1360244,80.19,67.0,64.0
2,Chennai,Deals Not Issued,371316,175810,173183,20.77,47.0,47.0
3,Chennai,Deals issused,1416529,954461,906179,79.23,67.0,64.0
4,Hyderabad,Deals Not Issued,1414226,740871,745877,40.66,52.0,53.0
5,Hyderabad,Deals issused,2064260,1470537,1413874,59.34,71.0,68.0


In [37]:
df_analysis2v1 = df\
                .groupby(['city_name', 'taxi_retention_segments', 'isIssued'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis2v1['customers distr %'] = round(df_analysis2v1['customers']*100.00/df_analysis2v1.groupby(['city_name', 'isIssued'])['customers'].transform('sum'), 2)
df_analysis2v1['nov retention %'] = round(df_analysis2v1['nov_retention']*100.00/df_analysis2v1['customers'])
df_analysis2v1['dec retention %'] = round(df_analysis2v1['dec_retention']*100.00/df_analysis2v1['customers'])
df_analysis2v1

,city_name,taxi_retention_segments,isIssued,customers,nov_retention,dec_retention,customers distr %,nov retention %,dec retention %
0,Bangalore,DORMANT,Deals Not Issued,311821,131807,135329,59.71,42.0,43.0
1,Bangalore,DORMANT,Deals issused,108219,50747,51545,5.12,47.0,48.0
2,Bangalore,ELITE,Deals Not Issued,4202,3614,3477,0.80,86.0,83.0
3,Bangalore,ELITE,Deals issused,293510,271971,263684,13.88,93.0,90.0
4,Bangalore,GOLD,Deals Not Issued,56304,39359,37501,10.78,70.0,67.0
5,Bangalore,GOLD,Deals issused,533408,372220,356720,25.23,70.0,67.0
6,Bangalore,HH,Deals Not Issued,37389,14343,12654,7.16,38.0,34.0
7,Bangalore,HH,Deals issused,299326,114409,103778,14.16,38.0,35.0
8,Bangalore,PLATINUM,Deals Not Issued,22824,18621,17737,4.37,82.0,78.0
9,Bangalore,PLATINUM,Deals issused,465606,392405,378114,22.02,84.0,81.0


In [36]:
df_analysis2v2 = df\
                .groupby(['city_name', 'taxi_regularity_segment', 'isIssued'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis2v2['customers distr %'] = round(df_analysis2v2['customers']*100.00/df_analysis2v2.groupby(['city_name', 'isIssued'])['customers'].transform('sum'), 2)
df_analysis2v2['nov retention %'] = round(df_analysis2v2['nov_retention']*100.00/df_analysis2v2['customers'])
df_analysis2v2['dec retention %'] = round(df_analysis2v2['dec_retention']*100.00/df_analysis2v2['customers'])
df_analysis2v2

,city_name,taxi_regularity_segment,isIssued,customers,nov_retention,dec_retention,customers distr %,nov retention %,dec retention %
0,Bangalore,BI_WEEKLY,Deals Not Issued,145201,86330,85656,27.80,59.0,59.0
1,Bangalore,BI_WEEKLY,Deals issused,535336,354440,341884,25.32,66.0,64.0
2,Bangalore,DAILY,Deals Not Issued,8536,5907,5949,1.63,69.0,70.0
3,Bangalore,DAILY,Deals issused,230999,213212,206755,10.92,92.0,90.0
4,Bangalore,MONTHLY,Deals Not Issued,117422,54125,53985,22.48,46.0,46.0
5,Bangalore,MONTHLY,Deals issused,259900,130061,127154,12.29,50.0,49.0
6,Bangalore,RARE_NEED,Deals Not Issued,68368,26863,26362,13.09,39.0,39.0
7,Bangalore,RARE_NEED,Deals issused,152041,76647,74307,7.19,50.0,49.0
8,Bangalore,UNKNOWN,Deals Not Issued,119619,34550,33024,22.90,29.0,28.0
9,Bangalore,UNKNOWN,Deals issused,324999,121332,110679,15.37,37.0,34.0


In [30]:
df_analysis3 = df\
                .groupby(['city_name','isScratched'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis3['customers distr %'] = round(df_analysis3['customers']*100.00/df_analysis3.groupby(['city_name'])['customers'].transform('sum'), 2)
df_analysis3['nov retention %'] = round(df_analysis3['nov_retention']*100.00/df_analysis3['customers'])
df_analysis3['dec retention %'] = round(df_analysis3['dec_retention']*100.00/df_analysis3['customers'])
df_analysis3

,city_name,isScratched,customers,nov_retention,dec_retention,customers distr %,nov retention %,dec retention %
0,Bangalore,Deals Not Issued,522616,254753,251422,19.81,49.0,48.0
1,Bangalore,"Deals issused, not scratched",1990881,1313174,1265517,75.48,66.0,64.0
2,Bangalore,"Deals issused, scratched",124122,97794,94727,4.71,79.0,76.0
3,Chennai,Deals Not Issued,371316,175810,173183,20.77,47.0,47.0
4,Chennai,"Deals issused, not scratched",1355884,905253,859528,75.84,67.0,63.0
5,Chennai,"Deals issused, scratched",60645,49208,46651,3.39,81.0,77.0
6,Hyderabad,Deals Not Issued,1414226,740871,745877,40.66,52.0,53.0
7,Hyderabad,"Deals issused, not scratched",1988710,1409475,1355358,57.17,71.0,68.0
8,Hyderabad,"Deals issused, scratched",75550,61062,58516,2.17,81.0,77.0


In [35]:
df_analysis3v1 = df\
                .groupby(['city_name', 'taxi_retention_segments', 'isScratched'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis3v1['customers distr %'] = round(df_analysis3v1['customers']*100.00/df_analysis3v1.groupby(['city_name', 'isScratched'])['customers'].transform('sum'), 2)
df_analysis3v1['nov retention %'] = round(df_analysis3v1['nov_retention']*100.00/df_analysis3v1['customers'])
df_analysis3v1['dec retention %'] = round(df_analysis3v1['dec_retention']*100.00/df_analysis3v1['customers'])
df_analysis3v1

,city_name,taxi_retention_segments,isScratched,customers,nov_retention,dec_retention,customers distr %,nov retention %,dec retention %
0,Bangalore,DORMANT,Deals Not Issued,311821,131807,135329,59.71,42.0,43.0
1,Bangalore,DORMANT,"Deals issused, not scratched",104968,49038,49792,5.27,47.0,47.0
2,Bangalore,DORMANT,"Deals issused, scratched",3251,1709,1753,2.62,53.0,54.0
3,Bangalore,ELITE,Deals Not Issued,4202,3614,3477,0.80,86.0,83.0
4,Bangalore,ELITE,"Deals issused, not scratched",254875,235696,228423,12.81,92.0,90.0
5,Bangalore,ELITE,"Deals issused, scratched",38635,36275,35261,31.13,94.0,91.0
6,Bangalore,GOLD,Deals Not Issued,56304,39359,37501,10.78,70.0,67.0
7,Bangalore,GOLD,"Deals issused, not scratched",507486,353911,339027,25.50,70.0,67.0
8,Bangalore,GOLD,"Deals issused, scratched",25922,18309,17693,20.89,71.0,68.0
9,Bangalore,HH,Deals Not Issued,37389,14343,12654,7.16,38.0,34.0


In [42]:
def resample(df):

    # Group by city_name and isScratched
    grouped = df.groupby(['city_name', 'isScratched'])
    
    # Function to resample within each group while maintaining stratification on segments
    def stratified_resample(group):
        # Stratify based on taxi_retention_segments and taxi_regularity_segment
        return group.groupby(['taxi_retention_segments', 'taxi_regularity_segment'], group_keys=False).apply(lambda x: x.sample(frac=1, random_state=42))
    
    # Apply the stratified resampling function to each group
    resampled_df = grouped.apply(stratified_resample)
    
    # Reset index to clean up the multi-level index from groupby
    resampled_df = resampled_df.reset_index(drop=True)

    return resampled_df

In [ ]:
resampled_df = resample(df)

In [44]:
resampled_analysisv1 = resampled_df\
                .groupby(['city_name', 'taxi_retention_segments', 'isScratched'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    nov_retention = ('customer_id', lambda x: x[df['nov_retention'] == 1].nunique()),
                    dec_retention = ('customer_id', lambda x: x[df['dec_retention'] == 1].nunique())
                    )\
                .reset_index()

resampled_analysisv1['customers distr %'] = round(resampled_analysisv1['customers']*100.00/resampled_analysisv1.groupby(['city_name', 'isScratched'])['customers'].transform('sum'), 2)
resampled_analysisv1['nov retention %'] = round(resampled_analysisv1['nov_retention']*100.00/resampled_analysisv1['customers'])
resampled_analysisv1['dec retention %'] = round(resampled_analysisv1['dec_retention']*100.00/resampled_analysisv1['customers'])
resampled_analysisv1

,city_name,taxi_retention_segments,isScratched,customers,nov_retention,dec_retention,customers distr %,nov retention %,dec retention %
0,Bangalore,DORMANT,Deals Not Issued,311821,197213,190990,59.71,63.0,61.0
1,Bangalore,DORMANT,"Deals issused, not scratched",104968,66658,64743,5.27,64.0,62.0
2,Bangalore,DORMANT,"Deals issused, scratched",3251,2037,2018,2.62,63.0,62.0
3,Bangalore,ELITE,Deals Not Issued,4202,2704,2619,0.80,64.0,62.0
4,Bangalore,ELITE,"Deals issused, not scratched",254875,161484,156713,12.81,63.0,61.0
5,Bangalore,ELITE,"Deals issused, scratched",38635,24350,23601,31.13,63.0,61.0
6,Bangalore,GOLD,Deals Not Issued,56304,35618,34534,10.78,63.0,61.0
7,Bangalore,GOLD,"Deals issused, not scratched",507486,320953,310863,25.50,63.0,61.0
8,Bangalore,GOLD,"Deals issused, scratched",25922,16492,15923,20.89,64.0,61.0
9,Bangalore,HH,Deals Not Issued,37389,23609,22929,7.16,63.0,61.0


In [ ]:
resampled_df.to_parquet('/Users/E2074/local-datasets/customer/ads-monetisation/deals-impact/resampled_data.parquet', index=False)